In [ ]:
# ============================================
# FINAL PROJECT ANALYSIS NOTEBOOK
# NBA PLAYER SCORING PROJECT
# ============================================

# STEP 1: Upload file
from google.colab import files
uploaded = files.upload()

Saving all_seasons.csv to all_seasons.csv


In [ ]:
import pandas as pd

df = pd.read_csv("all_seasons.csv")

df.head()

,Unnamed: 0,player_name,team_abbreviation,age,player_height,player_weight,college,country,draft_year,draft_round,...,pts,reb,ast,net_rating,oreb_pct,dreb_pct,usg_pct,ts_pct,ast_pct,season
0,0,Randy Livingston,HOU,22.0,193.04,94.800728,Louisiana State,USA,1996,2,...,3.9,1.5,2.4,0.3,0.042,0.071,0.169,0.487,0.248,1996-97
1,1,Gaylon Nickerson,WAS,28.0,190.50,86.182480,Northwestern Oklahoma,USA,1994,2,...,3.8,1.3,0.3,8.9,0.030,0.111,0.174,0.497,0.043,1996-97
2,2,George Lynch,VAN,26.0,203.20,103.418976,North Carolina,USA,1993,1,...,8.3,6.4,1.9,-8.2,0.106,0.185,0.175,0.512,0.125,1996-97
3,3,George McCloud,LAL,30.0,203.20,102.058200,Florida State,USA,1989,1,...,10.2,2.8,1.7,-2.7,0.027,0.111,0.206,0.527,0.125,1996-97
4,4,George Zidek,DEN,23.0,213.36,119.748288,UCLA,USA,1995,1,...,2.8,1.7,0.3,-14.1,0.102,0.169,0.195,0.500,0.064,1996-97


In [ ]:
df.columns

Index(['Unnamed: 0', 'player_name', 'team_abbreviation', 'age',
       'player_height', 'player_weight', 'college', 'country', 'draft_year',
       'draft_round', 'draft_number', 'gp', 'pts', 'reb', 'ast', 'net_rating',
       'oreb_pct', 'dreb_pct', 'usg_pct', 'ts_pct', 'ast_pct', 'season'],
      dtype='object')

In [ ]:
df['ppg'] = df['pts'] / df['gp']

df[['player_name','pts','gp','ppg']].head()

,player_name,pts,gp,ppg
0,Randy Livingston,3.9,64,0.060937
1,Gaylon Nickerson,3.8,4,0.950000
2,George Lynch,8.3,41,0.202439
3,George McCloud,10.2,64,0.159375
4,George Zidek,2.8,52,0.053846


In [ ]:
import pandas as pd

df = pd.read_csv("all_seasons.csv")
df['ppg'] = df['pts']
df = df[df['gp'] >= 20]

df[['player_name','gp','pts','ppg','reb','ast']].head()

,player_name,gp,pts,ppg,reb,ast
0,Randy Livingston,64,3.9,3.9,1.5,2.4
2,George Lynch,41,8.3,8.3,6.4,1.9
3,George McCloud,64,10.2,10.2,2.8,1.7
4,George Zidek,52,2.8,2.8,1.7,0.3
5,Gerald Wilkins,80,10.6,10.6,2.2,2.2


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

analysis_df = df[['ppg','reb','ast','age','net_rating']].dropna().copy()

print("Rows used:", analysis_df.shape[0])
print("\nSummary stats:")
print(analysis_df.describe().round(2))

X = analysis_df[['reb','ast','age','net_rating']]
y = analysis_df['ppg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("\nR2 Score:", round(r2_score(y_test, pred), 3))
print("\nCoefficients:")
for col, coef in zip(X.columns, model.coef_):
    print(f"{col}: {coef:.3f}")

Rows used: 10720

Summary stats:
            ppg       reb       ast       age  net_rating
count  10720.00  10720.00  10720.00  10720.00    10720.00
mean       9.22      3.93      2.04     27.17       -1.09
std        5.93      2.45      1.85      4.33        6.44
min        0.30      0.10      0.00     18.00      -40.00
25%        4.70      2.10      0.80     24.00       -5.30
50%        7.80      3.30      1.40     27.00       -0.80
75%       12.50      5.10      2.70     30.00        3.20
max       36.10     16.10     11.70     43.00       19.50

R2 Score: 0.64

Coefficients:
reb: 1.119
ast: 1.783
age: -0.102
net_rating: 0.055


In [ ]:
from scipy.stats import f_oneway

anova_df = df[['pts', 'draft_round']].dropna().copy()
anova_df['draft_round'] = anova_df['draft_round'].astype(str)

group1 = anova_df[anova_df['draft_round'] == '1']['pts']
group2 = anova_df[anova_df['draft_round'] == '2']['pts']
group3 = anova_df[~anova_df['draft_round'].isin(['1', '2'])]['pts']

print("Round 1 players:", len(group1))
print("Round 2 players:", len(group2))
print("Other/undrafted players:", len(group3))

f_stat, p_value = f_oneway(group1, group2, group3)

print("\nANOVA F-statistic:", round(f_stat, 3))
print("ANOVA p-value:", round(p_value, 5))

Round 1 players: 6734
Round 2 players: 2417
Other/undrafted players: 1569

ANOVA F-statistic: 662.188
ANOVA p-value: 0.0


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from scipy.stats import f_oneway

analysis_df = df[['ppg','reb','ast','age','net_rating']].dropna().copy()
X = analysis_df[['reb','ast','age','net_rating']]
y = analysis_df['ppg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
pred = model.predict(X_test)

anova_df = df[['pts','draft_round']].dropna().copy()
anova_df['draft_round'] = anova_df['draft_round'].astype(str)

group1 = anova_df[anova_df['draft_round'] == '1']['pts']
group2 = anova_df[anova_df['draft_round'] == '2']['pts']
group3 = anova_df[~anova_df['draft_round'].isin(['1', '2'])]['pts']

f_stat, p_value = f_oneway(group1, group2, group3)

print("R2 Score:", round(r2_score(y_test, pred), 3))
print("ANOVA p-value:", round(p_value, 5))

R2 Score: 0.64
ANOVA p-value: 0.0
